# 🕵️ Testando Positivo (parte 1): o detector de mentiras

Este é o primeiro notebook de um projeto que vai atravessar as Aulas 5 a 8. Em cada aula vamos atacar o **mesmo problema** com uma ferramenta matemática diferente. Aqui na Aula 5, vamos simular no computador a mesma dinâmica que fizemos ao vivo em sala com o detector de mentiras — só que em vez de uma turma de 50 pessoas, vamos simular 100.000 "funcionários" de uma vez, para ver o resultado com bem mais precisão.

Só executar e ler, sem entrega.

## 1. Simulando o cenário do detector de mentiras

O cenário (o mesmo da atividade em sala): 10% dos funcionários de uma loja são ladrões. O detector de mentiras acerta com probabilidade 0,8 tanto para ladrões (flagra) quanto para honestos (libera).

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # semente fixa só para o resultado ser sempre igual neste notebook
n_funcionarios = 100_000

eh_ladrao = rng.random(n_funcionarios) < 0.10
prob_falhar_no_teste = np.where(eh_ladrao, 0.8, 0.2)  # ladrão falha c/ 0.8; honesto falha (erro) c/ 0.2
falhou = rng.random(n_funcionarios) < prob_falhar_no_teste

print(f"Total de funcionários simulados: {n_funcionarios}")
print(f"Quantos são realmente ladrões: {eh_ladrao.sum()} ({eh_ladrao.mean():.1%})")
print(f"Quantos falharam no teste: {falhou.sum()} ({falhou.mean():.1%})")

## 2. A pergunta dos "gerentes": quem falhou é ladrão?

Assim como na sala de aula, vamos contar quantos dos que falharam no teste são de fato ladrões.

In [ ]:
falharam_e_sao_ladroes = (falhou & eh_ladrao).sum()
total_falharam = falhou.sum()
fracao_ladroes_entre_falharam = falharam_e_sao_ladroes / total_falharam

print(f"Dos {total_falharam} que falharam no teste, {falharam_e_sao_ladroes} são realmente ladrões.")
print(f"Fração de ladrões entre os que falharam: {fracao_ladroes_entre_falharam:.1%}")

**Com 100.000 funcionários simulados, o resultado é bem mais estável do que numa turma de 50 pessoas**: cerca de 31% dos que falham no teste são realmente ladrões — os outros quase 70% são honestos que o teste errou. Isso é a **probabilidade condicional invertida**: P(falhou | ladrão) = 0,8 é bem diferente de P(ladrão | falhou) ≈ 0,31. Vamos formalizar essa conta com o Teorema de Bayes na Aula 6.

## 3. O exemplo da urna, da aula: probabilidade condicional

Agora o exemplo dos slides: uma urna com 2 bolas brancas e 3 vermelhas. Sorteamos uma bola, sem devolver, e depois outra. Qual a probabilidade de a segunda ser vermelha, dado que a primeira foi branca?

In [ ]:
bolas = np.array(["branca", "branca", "vermelha", "vermelha", "vermelha"])
n_simulacoes = 200_000

# embaralha as 5 bolas em cada uma das 200.000 simulações de uma vez só (bem mais rápido que um loop)
chaves_aleatorias = rng.random((n_simulacoes, 5))
ordem = np.argsort(chaves_aleatorias, axis=1)
primeira = bolas[ordem[:, 0]]
segunda = bolas[ordem[:, 1]]

primeira_e_branca = primeira == "branca"
probabilidade_simulada = (segunda[primeira_e_branca] == "vermelha").mean()

print(f"P(2ª vermelha | 1ª branca), simulado: {probabilidade_simulada:.3f}")
print(f"Valor teórico: 3/4 = {3/4:.3f}")

Depois que a primeira bola branca sai (e não volta pra urna), sobram 4 bolas: 1 branca e 3 vermelhas — por isso a probabilidade da segunda ser vermelha sobe para 3/4, maior que a proporção original de vermelhas na urna (3/5).

## 4. Isso é independência? Vamos comparar com reposição

Se colocássemos a primeira bola de volta na urna antes de sortear a segunda (**com reposição**), a cor da segunda bola deixaria de depender da primeira. Vamos conferir.

In [ ]:
# com reposição: cada sorteio é independente, a urna sempre volta a ter 2 brancas e 3 vermelhas
primeira_reposicao = rng.choice(bolas, size=n_simulacoes)
segunda_reposicao = rng.choice(bolas, size=n_simulacoes)

primeira_branca_reposicao = primeira_reposicao == "branca"
probabilidade_com_reposicao = (segunda_reposicao[primeira_branca_reposicao] == "vermelha").mean()

print(f"Com reposição — P(2ª vermelha | 1ª branca): {probabilidade_com_reposicao:.3f}")
print(f"Compare com P(vermelha) simples, sem condicionar em nada: {3/5:.3f}")

Com reposição, as duas probabilidades ficam praticamente iguais (~0,6) — sinal de que os dois eventos são **independentes** nesse caso. Sem reposição (seção 3), elas eram diferentes (0,75 vs. 0,6) — sinal de **dependência**. É exatamente a definição de independência vista na aula: P(A|B) = P(A).

## Para fechar (por enquanto)

Guardamos o número **31%** — a fração de ladrões entre os que falham no teste. Na Aula 6 vamos calcular esse mesmo número usando o Teorema de Bayes, sem precisar simular nada, e ver que bate certinho com o que a simulação encontrou aqui.